# 03 · Full refit against `src_next`: tmQM + LnQM (single unattended run; prepared, NOT yet run)

Everything this notebook depends on is **native in `src_next/`** (no runtime patching, so spawn-based worker pools on Windows are fine).
`src/` and its `constraints.json` are never touched; the new DB is only ever copied into `src_next/jacobik/` (last cell, gated).

**What changed in `src_next` (and therefore in the statistics):**
1. **Q6**: pairs within 3 bonds on the FULL graph (H and metal-donor edges included) are excluded, in extraction *and* in `ik/geometry.py` (shared function). Metal pairs are now fitted (`(M,X)` plus pooled `(period_group,X)` keys).
2. **Donor lumps split**: `C_cyanide`/`C_isocyanide`; `N_ammonia/primary/secondary/tertiary_amine`, `N_amido`, `N_imido`; `O_water/O_hydroxide/O_oxo/O_alcohol/O_heteroxo` (the old `O_hydroxo` meant three different things).
3. **Q4 key** now includes the substituent count `k`; **Q5 flank** choice is canonical (element, bond type, degree, index last) and H-flanked rotors are extracted (extractor and runtime use one function); **haptic Q2 distance class**, donor labels, trans gating (`other`/linear/CN<=2 -> none) now come from one set of key functions (`key_builder`) used by the extractor, the scorer and the engine's Q1 sampler.
4. **Q1 keys use the metal OXIDATION STATE**, not the net charge. Training labels: tmQM = metal bracket charge in the SMILES (+ neutral hydrides, - nitro N([O-])[O-] mis-charges), accepted only when the SMILES total charge equals the header charge and the state is plausible for the metal; LnQM = Ln(III) when the header spin equals the Hund f^n count. Unreliable labels are `OS_UNKNOWN` (99) and only feed OS-free levels. Query time: `ComplexBuilder(..., oxidation_state=None)` deduces `net charge - sum(ionic ligand charges)` (logged; low confidence -> unknown), or take `oxidation_state=` explicitly.
5. **One ladder module** (`stats/ladders.py`) generates both the expander's ancestor keys and the lookup's candidates (the D1-D4 inconsistencies cannot recur). Q1 ordering is a *variant* chosen on held-out complexes; state-keeping variants are preferred when within 0.01 log-lik of the best.
6. **Switchable Shannon fallback** (`StatsLookup(shannon=True)` / `solve(shannon=True)`): only for (metal, OS, spin) states with no data; shifts a data-rich reference state by the Shannon radius difference. Validated here on held-out tmQM/LnQM states, and on DOE Fe2/Fe3 (never fitted on it).
7. Fitter fits are seeded per key; extractor errors are counted per quantity and reported; reservoirs are seeded.
8. **Joint donor orientation S_orient (quantity `QO`, 2026-09-24)** replaces Q3 + Q4 at solve time whenever the DB has QO fits (`solve(orientation_model='auto')`). Per kappa donor with k = 1-3 substituents: g = sorted cos(M-D-X_i) + cos(X_i-D-X_j); full-covariance GMM per ladder key (k, chelate ring size, subclass, metal, geometry, anion bucket). H-bonded donors (H...acceptor < 2.3 A) are excluded from the fit. Extracted by the same `extract_split` pass (`observations_QO_<split>.parquet`), fitted in section 6b, stored as `QO:` entries in `constraints_refit.json` (older code ignores them).

Sections: 0 config - 1 oxidation-label audit - 2 clean + extract - 3 sanity - 4 ladder selection - 5 Shannon hold-out - 6 full fit - 6b S_orient (hold-out check + fit) - 7 audits - 8 DOE validation - 9 verdict + gated install.

In [1]:
import sys, os, json, time, glob, re, shutil, logging, warnings, subprocess
os.environ["MKL_THREADING_LAYER"] = "SEQUENTIAL"
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src_next'/'jacobik').exists())
SRC_NEXT = ROOT/'src_next'; SRC_OLD = ROOT/'src'
sys.path.insert(0, str(SRC_NEXT)); sys.path.insert(0, str(ROOT/'refit_next'))
from rdkit import RDLogger; RDLogger.DisableLog('rdApp.*')
import refit_lib as R
from jacobik.stats import ladders as L
assert (SRC_NEXT/'jacobik'/'stats'/'ladders.py').exists()
RUN_HEAVY = True             # False: only reload cached outputs
N_WORKERS = 12
SHARD_SIZE = 4000
RUN_LOMO  = False            # optional leave-metal-out ladder test
INSTALL   = False            # False | True | 'if_pass'  (copies the new DB into src_next/jacobik only)
Q1_VARIANT_OVERRIDE = None   # None = automatic; or one of ladders.Q1_VARIANTS
TM_DIR    = ROOT/'data'/'refit'/'source'/'tmQM'
LN_SRC    = ROOT/'data'/'refit'/'source'/'lnQM'      # exporter output (Angstrom, unfiltered)
LN_XYZ_IN_BOHR = False
REFIT     = ROOT/'data'/'refit'
LN_CLEAN  = REFIT/'clean'/'lnQM'
TM_CLEAN  = REFIT/'clean'/'tmQM'
WORK      = REFIT; OBS = WORK/'observations'; HOLD = WORK/'reports'/'holdout'; FINAL = WORK/'final'
for d in (OBS, HOLD, FINAL): d.mkdir(parents=True, exist_ok=True)
QS = ['Q1','Q2','Q3','Q4','Q5','Q6']
SPLITS = [('X1',TM_CLEAN),('X2',TM_CLEAN),('X3',TM_CLEAN),('LNf',LN_CLEAN)]
print(ROOT)

c:\Users\alanc\sus folder\CCCSEF\Jacobik v3


## 1. Oxidation-state label audit (fast, no geometry)
Round-trips the labeller over every tmQM header+SMILES. Look at: reject reasons, and per-metal histograms (a state that is chemically odd for that metal is a SMILES artefact; it only ever feeds its own thin key and the OS-free levels).

In [2]:
AUD = R.os_label_audit(TM_DIR, splits=('X1','X2','X3'))
print(AUD['reasons'])
print(pd.DataFrame(AUD['histogram']).fillna(0).astype(int).T.loc[[m for m in ['Sc','Ti','V','Cr','Mn','Fe','Co','Ni','Cu','Zn','Mo','Ru','Rh','Pd','W','Re','Os','Ir','Pt','Au'] if m in AUD['histogram']]].to_string())

{'no_smiles_or_metal': 7692, 'ok': 100315, 'implausible': 359, 'charge_mismatch': 175}
      1     2     3   -3   -2   -1     0     4     5     6   -4    7    8    9    10   11   12
Sc    28     9   409    0    0    0     0     0     0     0    0    0    0    0    0    0    0
Ti     3   223    12    0    0    0    10  2821     0     0    0    0    0    0    0    0    0
V     33     1   317    0    0    5     2     0   986     0    0    0    0    0    0    0    0
Cr    14   336     2    0    0    1  1475    89     0   114    0    0    0    0    0    0    0
Mn  1193    15   702    0    1   39    45    12    78     0    0   13    0    0    0    0    0
Fe   170  3690    23    3    3   11   928   159     0    37    0    0    4    0    0    0    0
Co   841    39  2344    1    0   45    27    14    84     2    0   24    0   11    0    0    0
Ni    87  8487    60    0   11    3   872   369     1    79    1    0   27    0    9    0    0
Cu  3055    18   322    0    0   37     8     2    50     

## 2. Clean contacts (tmQM AND LnQM), then extract everything (heavy)
Thresholds are derived from the data (`contact_filter_params.json`); every dropped contact is kept with its reason. Extraction reports per-quantity errors; a non-zero count is printed loudly.

In [3]:
CF = WORK/'reports'/'contacts'; CF.mkdir(parents=True, exist_ok=True); SHARD_DIR = OBS/'shards'
SPECS = [dict(name=sp,group='TM',src=TM_DIR,src_split=sp,out=TM_CLEAN,split=sp,scale=1.0) for sp in ('X1','X2','X3')] + \
        [dict(name='LNf',group='LN',src=LN_SRC,src_split='LN',out=LN_CLEAN,split='LNf',scale=(R._BOHR if LN_XYZ_IN_BOHR else 1.0))]
if RUN_HEAVY or not (CF/'shards.json').exists():
    t=time.time(); THR=R.prepare_clean(SPECS, CF, n_workers=N_WORKERS, shard_size=SHARD_SIZE); print('contact cleaning + sharding',round(time.time()-t),'s')
else:
    THR=json.load(open(CF/'contact_filter_params.json'))

SHARDS=json.load(open(CF/'shards.json')); print(len(SHARDS),'shards')

  cleaned 5/34 shards
  cleaned 10/34 shards
  cleaned 15/34 shards
  cleaned 20/34 shards
  cleaned 25/34 shards
  cleaned 30/34 shards
  cleaned 34/34 shards
contact cleaning + sharding 51 s
34 shards


In [4]:
EXTRACT_STATS={}
if RUN_HEAVY or not all((OBS/f'observations_Q1_{sp}.parquet').exists() for sp,_ in SPLITS):
    t=time.time(); shard_stats=R.extract_shards(SHARDS, SHARD_DIR, N_WORKERS); print('extraction',round(time.time()-t),'s')

  extracted 1/34 X1s008: 2970 complexes, 153 ms/mol, errors=0 quantity_errors={'Q1': 0, 'Q2': 0, 'Q3': 0, 'Q4': 0, 'Q5': 0, 'Q6': 0}
  extracted 2/34 X2s001: 4000 complexes, 154 ms/mol, errors=0 quantity_errors={'Q1': 0, 'Q2': 0, 'Q3': 0, 'Q4': 0, 'Q5': 0, 'Q6': 0}
  extracted 3/34 X2s000: 4000 complexes, 162 ms/mol, errors=0 quantity_errors={'Q1': 0, 'Q2': 0, 'Q3': 0, 'Q4': 0, 'Q5': 0, 'Q6': 0}
  extracted 4/34 X1s007: 4000 complexes, 164 ms/mol, errors=0 quantity_errors={'Q1': 0, 'Q2': 0, 'Q3': 0, 'Q4': 0, 'Q5': 0, 'Q6': 0}
  extracted 5/34 X1s000: 4000 complexes, 183 ms/mol, errors=0 quantity_errors={'Q1': 0, 'Q2': 0, 'Q3': 0, 'Q4': 0, 'Q5': 0, 'Q6': 0}
  extracted 6/34 X2s002: 4000 complexes, 198 ms/mol, errors=0 quantity_errors={'Q1': 0, 'Q2': 0, 'Q3': 0, 'Q4': 0, 'Q5': 0, 'Q6': 0}
  extracted 7/34 X1s006: 4000 complexes, 201 ms/mol, errors=0 quantity_errors={'Q1': 0, 'Q2': 0, 'Q3': 0, 'Q4': 0, 'Q5': 0, 'Q6': 0}
  extracted 8/34 X2s009: 65 complexes, 323 ms/mol, errors=0 quantity_

In [5]:
if RUN_HEAVY or not all((OBS/f'observations_Q1_{sp}.parquet').exists() for sp,_ in SPLITS):
    for sp,_ in SPLITS:
        names=[s['shard'] for s in SHARDS if s['dataset']==sp]
        R.merge_shards(SHARD_DIR, OBS, sp, names)
        EXTRACT_STATS[sp]={'n_complexes':sum(shard_stats[n]['n_complexes'] for n in names),'n_errors':sum(shard_stats[n]['n_errors'] for n in names),
                           'quantity_errors':{q:sum(shard_stats[n]['quantity_errors'][q] for n in names) for q in QS}}
    print(json.dumps(EXTRACT_STATS,indent=1))

rows=[]
for sp,_ in SPLITS:
    n_c=max(1,EXTRACT_STATS.get(sp,{}).get('n_complexes') or pd.read_parquet(OBS/f'observations_Q1_{sp}.parquet',columns=['csd_code']).csd_code.nunique())
    rows.append({'split':sp,**{q:len(pd.read_parquet(OBS/f'observations_{q}_{sp}.parquet',columns=['csd_code'])) for q in QS},'complexes':n_c})
D=pd.DataFrame(rows).set_index('split'); print(D); print((D[QS].div(D['complexes'],axis=0)).round(2))

{
 "X1": {
  "n_complexes": 34970,
  "n_errors": 0,
  "quantity_errors": {
   "Q1": 0,
   "Q2": 0,
   "Q3": 0,
   "Q4": 0,
   "Q5": 0,
   "Q6": 0
  }
 },
 "X2": {
  "n_complexes": 36065,
  "n_errors": 0,
  "quantity_errors": {
   "Q1": 0,
   "Q2": 0,
   "Q3": 0,
   "Q4": 0,
   "Q5": 0,
   "Q6": 0
  }
 },
 "X3": {
  "n_complexes": 37506,
  "n_errors": 0,
  "quantity_errors": {
   "Q1": 0,
   "Q2": 0,
   "Q3": 0,
   "Q4": 0,
   "Q5": 0,
   "Q6": 0
  }
 },
 "LNf": {
  "n_complexes": 17269,
  "n_errors": 0,
  "quantity_errors": {
   "Q1": 0,
   "Q2": 0,
   "Q3": 0,
   "Q4": 0,
   "Q5": 0,
   "Q6": 0
  }
 }
}
           Q1      Q2      Q3     Q4      Q5      Q6  complexes
split                                                          
X1     165652  345665  142044  83039  365740  224853      34970
X2     178255  380551  157512  94072  342880  172267      36065
X3     148590  244638  125577  92815  347123  161683      37506
LNf    122287  393018  111652  59962   82942  123887      17269
    

In [6]:
# import sys, json
# from pathlib import Path
# import pandas as pd

# ROOT=Path(r'C:\Users\alanc\sus folder\CCCSEF\Jacobik v3')
# sys.path.insert(0,str(ROOT/'src_next'))
# sys.path.insert(0,str(ROOT/'refit_next'))
# import refit_lib as R

# OBS=ROOT/'data'/'refit'/'observations'; SHARD_DIR=OBS/'shards'
# QS=['Q1','Q2','Q3','Q4','Q5','Q6']
# SPLITS=[('X1',None),('X2',None),('X3',None),('LNf',None)]
# SHARDS=json.load(open(ROOT/'data'/'refit'/'reports'/'contacts'/'shards.json'))

# EXTRACT_STATS={}
# for sp,_ in SPLITS:
#     names=[s['shard'] for s in SHARDS if s['dataset']==sp]
#     R.merge_shards(SHARD_DIR, OBS, sp, names)
#     EXTRACT_STATS[sp]={'n_complexes':sum(R.count_complexes(Path(s['dir']),s['shard']) for s in SHARDS if s['dataset']==sp),
#                        'n_errors':0,'quantity_errors':{q:0 for q in QS}}
# print(json.dumps(EXTRACT_STATS,indent=1))

# rows=[]
# for sp,_ in SPLITS:
#     n_c=EXTRACT_STATS[sp]['n_complexes']
#     rows.append({'split':sp,**{q:len(pd.read_parquet(OBS/f'observations_{q}_{sp}.parquet',columns=['csd_code'])) for q in QS},'complexes':n_c})
# D=pd.DataFrame(rows).set_index('split'); print(D); print((D[QS].div(D['complexes'],axis=0)).round(2))

In [7]:
# import sys
# from pathlib import Path

# ROOT = Path(r"C:\Users\alanc\sus folder\CCCSEF\Jacobik v3")
# sys.path.insert(0, str(ROOT / "src_next"))

# from jacobik.stats.extractor import extract_split

# st = extract_split(
#     data_dir=ROOT / "data" / "refit" / "clean" / "tmQM",
#     split="X1s007",
#     out_dir=ROOT / "data" / "refit" / "observations" / "shards",
#     progress=True,
#     progress_file=ROOT / "data" / "refit" / "observations" / "X1s007.progress.txt",
# )
# print(st)

### 2a. What the contact cleaning did

In [8]:
print('through-bond angle cut (deg):',THR['theta'],'|',THR['theta_info'])
print('second-sphere cutoffs (A), by metal period/lanthanide series and donor element:',THR.get('cutoffs') or {k:v for k,v in THR.items() if 'cut' in k})
dr=pd.concat([pd.read_parquet(p) for p in sorted(CF.glob('dropped_contacts_*.parquet'))],ignore_index=True) if list(CF.glob('dropped_contacts_*.parquet')) else pd.DataFrame(columns=['reason'])
print(dr.groupby('reason').size() if len(dr) else 'no dropped-contact files')

through-bond angle cut (deg): 147.5 | {'n': 84723, 'method': 'valley', 'theta': 147.5, 'linear_mode': 177.5, 'min_density': 502.30674151024834, 'linear_peak': 6593.675487988054, 'eta_peak': 12000.767223915782}
second-sphere cutoffs (A), by metal period/lanthanide series and donor element: {'3d|C': 3.5250000000000017, '3d|N': 3.4250000000000016, '3d|S': 3.925000000000002, '4d|C': 3.6750000000000016, '5d|C': 3.6250000000000018, '5d|F': 2.3250000000000006, '5d|Si': 2.8750000000000013, 'Ln|N': 3.825000000000002, 'Ln|O': 3.575000000000002, 'Ln|S': 3.8750000000000018}
reason
second_sphere     6595
through_bond     52455
dtype: int64


## 3. Sanity of the extracted statistics (before any fitting)

In [9]:
import ast
q1=pd.concat([pd.read_parquet(OBS/f'observations_Q1_{sp}.parquet').assign(dataset=sp) for sp,_ in SPLITS],ignore_index=True)
kk=q1.key.map(ast.literal_eval); q1['metal']=kk.map(lambda k:k[0]); q1['os']=kk.map(lambda k:k[2]); q1['sub']=kk.map(lambda k:k[3])
print('OS coverage (share of Q1 rows with a known oxidation state):'); print(q1.assign(known=q1.os!=L.OS_UNKNOWN).groupby('dataset').known.mean().round(3).to_string())
sp_=['O_water','O_hydroxide','O_oxo','O_alcohol','O_heteroxo','C_cyanide','C_isocyanide','N_ammonia','N_primary_amine','N_secondary_amine','N_tertiary_amine','N_amido','N_imido']
print(q1[q1['sub'].isin(sp_)].groupby(['sub','dataset']).value.agg(['median','std','size']).round(3).to_string())
# Q6: metal pairs present, and nothing at impossible distances
q6=pd.concat([pd.read_parquet(OBS/f'observations_Q6_{sp}.parquet').assign(dataset=sp) for sp,_ in SPLITS],ignore_index=True)
q6['pair']=q6.key
print('Q6 rows',len(q6),'| distinct pair keys',q6.pair.nunique(),'| pairs involving a metal:',int(q6.apply(lambda r:r.metal in r.key,axis=1).sum()))
print(q6.groupby('pair').value.quantile(0.05).sort_values().head(12).round(2).to_string())

OS coverage (share of Q1 rows with a known oxidation state):
dataset
LNf    1.000
X1     0.870
X2     0.938
X3     0.956
                           median    std   size
sub               dataset                      
C_cyanide         LNf       2.548  0.268   2848
                  X1        1.980  0.093    199
                  X2        1.904  0.078    295
                  X3        1.955  0.095    281
C_isocyanide      X1        1.923  0.120    853
                  X2        1.906  0.081    905
                  X3        1.896  0.064    590
N_amido           LNf       2.801  0.276    102
                  X1        2.035  0.184   7504
                  X2        1.977  0.176   2760
                  X3        1.974  0.239   2591
N_ammonia         LNf       2.557  0.089   6173
                  X1        2.168  0.110     97
                  X2        2.156  0.125    156
                  X3        2.122  0.106    259
N_imido           LNf       3.662  0.464      7
               

## 4. Select the Q1 fallback ordering on held-out COMPLEXES
Every variant is scored on the same held-out complexes through the same `ladders` module the runtime uses. State-keeping variants (which keep spin/OS in their specific levels) win ties within 0.01.

In [3]:
from jacobik.stats.expander import expand_quantity
from jacobik.stats.fitter import fit_all
obs_q1=[OBS/f'observations_Q1_{sp}.parquet' for sp,_ in SPLITS]
if RUN_HEAVY and not (HOLD/'c_train_Q1.json').exists():
    print('train/held-out rows',R.split_observations(obs_q1,'Q1',HOLD/'cx'))
    print(expand_quantity(HOLD/'cx',HOLD/'cx'/'exp',quantities=['Q1']))
    print(fit_all(HOLD/'cx'/'exp',HOLD/'c_train_Q1.json',quantities=['Q1'],n_workers=N_WORKERS))
EV=R.evaluate_q1_ladders(HOLD/'c_train_Q1.json',HOLD/'cx'/'heldout_Q1.parquet')
EV.drop(columns='levels').round(3)

,variant,n_min,group,n,unresolved,mean_loglik,MAE_mean
0,orig,30,Ln,11908,0,0.728,0.104
1,orig,30,TM,48743,0,0.927,0.070
2,orig,100,Ln,11908,0,0.703,0.110
3,orig,100,TM,48743,0,0.898,0.079
4,m_sub_early,30,Ln,11908,0,0.729,0.104
5,m_sub_early,30,TM,48743,0,0.959,0.066
6,m_sub_early,100,Ln,11908,0,0.705,0.110
7,m_sub_early,100,TM,48743,0,0.936,0.073
8,mo_state,30,Ln,11908,0,0.729,0.104
9,mo_state,30,TM,48743,0,0.959,0.066


In [4]:
PICK = R.pick_q1_variant(EV, n_min=30, tol=0.01); print(PICK['table'].round(3).to_string())
Q1_VARIANT = Q1_VARIANT_OVERRIDE or PICK['variant']
print('using Q1 ladder variant:',Q1_VARIANT,'(override)' if Q1_VARIANT_OVERRIDE else f"(automatic; raw best {PICK['best_raw']}, state-keeping preferred={PICK['preferred_state_keeping']})")
if RUN_LOMO:
    for held in (['Gd'],['Pm'],['Ho']):
        tag='lomo_'+'_'.join(held); d=HOLD/tag
        R.split_by_metal(obs_q1,'Q1',d,held); expand_quantity(d,d/'exp',quantities=['Q1']); fit_all(d/'exp',d/'c.json',quantities=['Q1'],n_workers=N_WORKERS)
        print(held); print(R.evaluate_q1_ladders(d/'c.json',d/'heldout_Q1.parquet').drop(columns='levels').round(3).to_string())

             loglik    MAE  unresolved  loglik_Ln  loglik_TM
variant                                                     
no_state      0.925  0.074         0.0      0.728      0.973
metal_first   0.914  0.073         0.0      0.729      0.960
m_sub_early   0.914  0.073         0.0      0.729      0.959
mo_state      0.914  0.073         0.0      0.729      0.959
cn_state      0.913  0.072         0.0      0.737      0.956
orig          0.888  0.077         0.0      0.728      0.927
using Q1 ladder variant: no_state (automatic; raw best no_state, state-keeping preferred=False)


## 5. Shannon fallback: held-out validation (no DOE)
For each (metal, spin, OS) state with data, its own keys are ignored; the state's mean is predicted (a) by the pooled state-free ladder, (b) by borrowing the best other state of the same (metal, donor, geometry) and shifting by the Shannon radius difference. `MAE_shannon < MAE_pooled` for most states => turn `shannon=True` on for the states the databases cannot cover.

In [5]:
SH = R.shannon_holdout_eval(HOLD/'cx'/'heldout_Q1.parquet', HOLD/'c_train_Q1.json')
print(SH.round(3).sort_values('n',ascending=False).head(40).to_string() if len(SH) else 'no state with enough held-out rows')
if len(SH): print('weighted MAE  pooled %.4f  shannon %.4f  | shannon better for %d/%d states'%(np.average(SH.MAE_pooled,weights=SH.n),np.average(SH.MAE_shannon,weights=SH.n),(SH.MAE_shannon<SH.MAE_pooled).sum(),len(SH)))

   metal  spin  OS     n  MAE_pooled  MAE_shannon
11    Ni     1   2  1067       0.143        0.204
15    Pd     1   2   887       0.050        0.056
17    Pt     1   2   482       0.046        0.056
9     Mo     1   6   329       0.030        0.090
24    Ti     1   4   276       0.053        0.441
18    Pt     1   4   240       0.062        0.099
5     Fe     1   2   232       0.062        0.162
3     Cu     1   1   189       0.042        0.060
8     Mo     1   4   126       0.091        0.161
12    Ni     1   4   124       0.140        0.148
26     V     1   5   110       0.076        0.162
19    Re     1   5    94       0.009        0.107
16    Pd     1   4    76       0.076        0.061
23    Ti     1   2    68       0.095        0.393
7     Mn     1   3    65       0.034        0.283
4     Cu     1   3    61       0.163        0.128
25     V     1   3    58       0.109        0.225
6     Fe     1   4    48       0.141        0.108
28     W     1   6    39       0.049        0.070


## 6. Full fit, all quantities, all data (heavy)

In [6]:
from jacobik.stats.aggregator import aggregate
from jacobik.stats.expander import expand_quantity
from jacobik.stats.fitter import fit_all
if RUN_HEAVY or not (FINAL/'constraints_refit.json').exists():
    for q in QS:
        for sp,_ in SPLITS: shutil.copy(OBS/f'observations_{q}_{sp}.parquet',FINAL/f'observations_{q}_{sp}.parquet')
    print(aggregate(FINAL,[sp for sp,_ in SPLITS],QS))
    t=time.time(); print(expand_quantity(FINAL,FINAL/'exp',quantities=QS),round(time.time()-t),'s')
    t=time.time(); print(fit_all(FINAL/'exp',FINAL/'constraints_refit.json',quantities=QS,n_workers=N_WORKERS),round(time.time()-t),'s')
json.dump({'q1_variant':Q1_VARIANT,'oxidation_keyed':True,'ladders':'jacobik.stats.ladders'},open(FINAL/'constraints_refit.meta.json','w'))
new=json.load(open(FINAL/'constraints_refit.json'))
print(pd.Series([k.split(':')[0] for k in new]).value_counts().to_string())

{'Q1': {'rows': 614784, 'missing_splits': [], 'elapsed_s': 0.6499451999552548, 'size_mb': 9.306474}, 'Q2': {'rows': 1363872, 'missing_splits': [], 'elapsed_s': 1.3129922000225633, 'size_mb': 17.654163}, 'Q3': {'rows': 536785, 'missing_splits': [], 'elapsed_s': 0.4975546000059694, 'size_mb': 7.230017}, 'Q4': {'rows': 329888, 'missing_splits': [], 'elapsed_s': 0.3308651999104768, 'size_mb': 4.48097}, 'Q5': {'rows': 1138685, 'missing_splits': [], 'elapsed_s': 0.9364971000468358, 'size_mb': 13.60152}, 'Q6': {'rows': 682690, 'missing_splits': [], 'elapsed_s': 0.5455939000239596, 'size_mb': 8.322641}}
{'Q1': {'in_rows': 614784, 'out_rows': 5193322, 'l0_rows': 413974, 'ancestor_rows': 4779348}, 'Q2': {'in_rows': 1363872, 'out_rows': 6199974, 'l0_rows': 1363872, 'ancestor_rows': 4836102}, 'Q3': {'in_rows': 536785, 'out_rows': 1031575, 'l0_rows': 536785, 'ancestor_rows': 494790}, 'Q4': {'in_rows': 329888, 'out_rows': 426759, 'l0_rows': 329888, 'ancestor_rows': 96871}, 'Q5': {'in_rows': 1138685,

## 6b. Joint donor orientation (S_orient, quantity QO)
Hold-out check first (fit on 90 % of complexes, score the other 10 %; log p per feature dimension, higher is better; H-bonded donors are excluded from fitting by design, so they score lower), then the final fit on all rows, merged into `constraints_refit.json` as `QO:` entries. See `src_next/jacobik/stats/orientation.py` and `math.md` section S5.

In [ ]:
QO = R.load_qo(OBS, [sp for sp,_ in SPLITS])
print(len(QO), 'donor rows;', QO.groupby('dataset').size().to_dict(), '| by k:', QO.groupby('k').size().to_dict())
print('H-bonded (excluded from fit):', int((QO.hb_min < 2.3).sum()))
EVQ_PATH = HOLD/'qo_holdout.json'
if RUN_HEAVY or not EVQ_PATH.exists():
    t=time.time(); EVQ = R.qo_holdout_eval(QO, mod=10, n_jobs=N_WORKERS); print('hold-out fit', round(time.time()-t), 's')
    json.dump({k: v for k, v in EVQ.items() if k != 'by_class'}, open(EVQ_PATH, 'w'), indent=1)
    EVQ['by_class'].to_csv(HOLD/'qo_holdout_by_class.csv')
else:
    EVQ = json.load(open(EVQ_PATH)); EVQ['by_class'] = pd.read_csv(HOLD/'qo_holdout_by_class.csv', index_col=0)
print(json.dumps(EVQ['loglik'], indent=1))
EVQ['by_class'].head(25).round(3)

In [ ]:
QO_FITS = FINAL/'orientation_fits.json'
if RUN_HEAVY or not QO_FITS.exists():
    t=time.time(); QO_ENTRIES = R.qo_fit_full(QO, n_jobs=N_WORKERS); print('full S_orient fit', round(time.time()-t), 's,', len(QO_ENTRIES), 'keys')
    json.dump(QO_ENTRIES, open(QO_FITS, 'w'))
else:
    QO_ENTRIES = json.load(open(QO_FITS))
print(R.add_entries_to_db(FINAL/'constraints_refit.json', QO_ENTRIES))
R.qo_level_audit(QO_ENTRIES)

## 7. Audits

In [7]:
# (a) fitted Q1 components beyond 3.0 A: second-sphere contamination that survived cleaning (should be empty or explainable)
FAR = R.far_component_audit(FINAL/'constraints_refit.json', far_A=3.0); print(len(FAR),'far components'); print(FAR.head(25).to_string())
# (b) Q6 wall positions for metal pairs (5th-percentile mu) and the effect of the topological exclusion on H..H / C..H
q6db={k[3:]:v for k,v in new.items() if k.startswith('Q6:')}
print(pd.DataFrame({k:{'mu':round(v['mu'],2),'sigma':round(v['sigma'],2),'n':v['n']} for k,v in q6db.items() if any(m in k for m in ('Fe','Pt','Gd','period4','lanthanide'))}).T.head(25).to_string())
# (c) lanthanide water / hydroxide from the resolved (m,sub) level
mean=lambda e: float(np.dot(e['weights'],e['means']))
for sub in ('O_water','O_hydroxide','O_oxo'):
    print(sub,{m:round(mean(new[f"Q1:{(m,sub)!r}"]),3) for m in R.LN if f"Q1:{(m,sub)!r}" in new})

2005 far components
                            key     n  weight   mean
109                ('C_donor',)  7872   0.072  3.534
94    ('lanthanide', 'N_donor')  4124   0.037  3.366
106    ('metal_any', 'P_donor')  3164   0.245  3.281
107                ('P_donor',)  3164   0.231  3.036
105    ('metal_any', 'P_donor')  3164   0.154  3.035
108                ('P_donor',)  3164   0.149  3.537
346           ('Pd', 'C_donor')  2836   0.054  3.020
1114       ('Nd', 3, 'N_donor')  2807   0.027  3.461
1113    ('Nd', 4, 3, 'N_donor')  2807   0.025  3.471
1115          ('Nd', 'N_donor')  2807   0.025  3.453
1377          ('Pm', 'N_donor')  2769   0.018  3.432
1375    ('Pm', 5, 3, 'N_donor')  2769   0.018  3.442
1376       ('Pm', 3, 'N_donor')  2769   0.018  3.438
533           ('Ho', 'N_donor')  2744   0.037  3.422
531     ('Ho', 5, 3, 'N_donor')  2744   0.033  3.418
532        ('Ho', 3, 'N_donor')  2744   0.034  3.419
1084       ('Sm', 3, 'N_donor')  2740   0.017  3.419
1083    ('Sm', 6, 3, 'N_do

## 8. DOE aqua validation (validation only, never fitted on)
Run as separate processes so the production `src` (baseline) and `src_next` never share an interpreter. Reports mean M-O error and the water **H-orientation** error (angle between O->M and the H-O-H bisector, conformer vs reference). The best conformer is the one with the highest score (the engine maximises).

In [8]:
def doe(src, db, tag, extra=()):
    out=WORK/f'doe_{tag}.json'
    cmd=[sys.executable, str(ROOT/'refit_next'/'doe_aqua.py'), '--src', str(src), '--db', str(db), '--out', str(out), '--doe-dir', str(next(iter(glob.glob(str(ROOT/'data'/'10.25584 - 3019314'/'*'/'opt-xyz'))))), *extra]
    subprocess.run(cmd, check=True, cwd=str(ROOT)); return json.load(open(out))
def summ(res):
    r=[x for x in res['rows'] if not x['system'].startswith('Fe')]; e=np.array([x['got']-x['ref'] for x in r]); th=np.array([x['tilt_got']-x['tilt_ref'] for x in r])
    return dict(n=len(r),MAE=np.abs(e).mean(),bias=e.mean(),sd=e.std(),corr=np.corrcoef([x['ref'] for x in r],[x['got'] for x in r])[0,1],tiltMAE_deg=np.nanmean(np.abs(th)))
DOE={}
DOE['baseline(src)']=doe(SRC_OLD, SRC_OLD/'jacobik'/'constraints.json', 'baseline')
DOE['refit']=doe(SRC_NEXT, FINAL/'constraints_refit.json', 'refit')
DOE['refit+shannon']=doe(SRC_NEXT, FINAL/'constraints_refit.json', 'refit_shannon', ['--shannon'])
print(pd.DataFrame({k:summ(v) for k,v in DOE.items()}).T.round(3).to_string())
print(pd.DataFrame({k:{f"{x['system']}/{x['cn']}":round(x['got']-x['ref'],3) for x in v['rows']} for k,v in DOE.items()}).to_string())

                  n    MAE   bias     sd   corr  tiltMAE_deg
baseline(src)  24.0  0.114 -0.093  0.087  0.507       20.240
refit          24.0  0.064  0.064  0.028  0.919       27.505
refit+shannon  24.0  0.064  0.064  0.028  0.919       27.505
       baseline(src)  refit  refit+shannon
Ce3/9         -0.231  0.040          0.040
Dy3/8         -0.056  0.107          0.107
Er3/8         -0.037  0.086          0.086
Er3/9         -0.076  0.029          0.029
Eu3/8         -0.112  0.072          0.072
Eu3/9         -0.140  0.034          0.034
Gd3/8         -0.091  0.099          0.099
Gd3/9         -0.128  0.034          0.034
Ho3/8         -0.048  0.064          0.064
Ho3/9         -0.086  0.028          0.028
La3/8          0.138  0.106          0.106
La3/9          0.107  0.091          0.091
Nd3/8         -0.153  0.071          0.071
Nd3/9         -0.187  0.027          0.027
Pm3/8         -0.150  0.074          0.074
Pm3/9         -0.170  0.037          0.037
Pr3/8         -0.175  0.1

In [ ]:
# Fe(H2O)6 2+/3+ : which Q1 level did the O_water query resolve at, with and without the Shannon shift
from jacobik.stats.stats import StatsLookup, load_db
DB=load_db(str(FINAL/'constraints_refit.json')); DB.q1_variant=Q1_VARIANT
for shannon in (False,True):
    st=StatsLookup(DB, shannon=shannon)
    for name,os_,sp in (('Fe2',2,5),('Fe3',3,6)):
        fd,rk,_=st.dist_trace('Q1',('Fe',sp,os_,'O_water','octahedral','O_water'))
        print(f"shannon={shannon} {name}: resolved {rk}  n={fd.n}  mean={np.dot(fd.weights,fd.means):.3f}  {st.last_note or ''}")
print(pd.DataFrame(DOE['refit']['fe_seeds']).round(3).to_string())

## 9. Verdict and gated install
The DB is only ever copied into `src_next/jacobik/` (with a backup). `src/` stays untouched, so the production DB keeps working with `src`.

In [ ]:
_b,_r,_s=summ(DOE['baseline(src)']),summ(DOE['refit']),summ(DOE['refit+shannon'])
FE=pd.DataFrame(DOE['refit']['fe_seeds'])
VERDICT={'doe_mae':{'baseline':round(_b['MAE'],4),'refit':round(_r['MAE'],4),'refit+shannon':round(_s['MAE'],4)},
         'tilt_mae_deg':{'baseline':round(_b['tiltMAE_deg'],1),'refit':round(_r['tiltMAE_deg'],1)},
         'doe_ok':bool(_r['MAE']<=_b['MAE']),'fe_seeds_ok':bool((abs(FE.FeO_mean-2.03)<0.15).all() and (FE.FeO_max<2.5).all()),
         'far_components':int(len(FAR)),'extract_errors':int(sum(v['n_errors']+sum(v['quantity_errors'].values()) for v in EXTRACT_STATS.values())) if EXTRACT_STATS else None}
VERDICT['pass']=VERDICT['doe_ok'] and VERDICT['fe_seeds_ok'] and not VERDICT['extract_errors']
print(json.dumps(VERDICT,indent=1)); print('VERDICT:', 'PASS' if VERDICT['pass'] else 'REVIEW (read the tables above; nothing installed)')
tgt=SRC_NEXT/'jacobik'/'constraints.json'; bak=tgt.with_name('constraints.pre_refit.json')
_do = INSTALL is True or (INSTALL=='if_pass' and VERDICT['pass'])
if _do:
    if tgt.exists() and not bak.exists(): shutil.copy(tgt,bak)
    shutil.copy(FINAL/'constraints_refit.json',tgt); shutil.copy(FINAL/'constraints_refit.meta.json',tgt.with_name('constraints.meta.json'))
    print('installed into src_next; to use it: put src_next on the path (or replace src once you are happy)')
else: print('not installed (INSTALL=%r, pass=%r)'%(INSTALL,VERDICT['pass']))